In [4]:
import numpy as np
import pandas as pd
import requests
import time
import folium
from folium.plugins import HeatMap

# --------------------------------------------------
# 1. Bayern-Raster erzeugen
# --------------------------------------------------

latitudes = np.arange(47.2, 50.6, 0.25)
longitudes = np.arange(8.9, 13.9, 0.25)

points = []

for lat in latitudes:
    for lon in longitudes:
        points.append((round(lat, 4), round(lon, 4)))

grid = pd.DataFrame(points, columns=["lat", "lon"])


# --------------------------------------------------
# 2. NASA POWER Daten abrufen
# --------------------------------------------------

def get_irradiance(lat, lon):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"

    params = {
        "parameters": "ALLSKY_SFC_SW_DWN",
        "community": "RE",
        "longitude": lon,
        "latitude": lat,
        "start": "20230101",
        "end": "20231231",
        "format": "JSON"
}

    try:
        response = requests.get(url, params=params, timeout=20)
        response.raise_for_status()

        data = response.json()
        irradiance = data["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"]

        annual_sum = sum(
            irradiance.values())

        return annual_sum

    except Exception as e:
        print(f"Fehler bei {lat}, {lon}: {e}")

        return None


# --------------------------------------------------
# 3. Alle Rasterpunkte auswerten
# --------------------------------------------------

results = []

for _, row in grid.iterrows():
    print(f"Lade {row.lat}, {row.lon}")

    annual_irradiance = get_irradiance(row.lat, row.lon)

    results.append({
        "lat": row.lat,
        "lon": row.lon,
        "annual_irradiance": annual_irradiance
})

    time.sleep(0.3)


# --------------------------------------------------
# 4. Ranking erstellen
# --------------------------------------------------

ranking = pd.DataFrame(results)

ranking = ranking.dropna(subset=["annual_irradiance"])

# Werte runden, damit fast gleiche NASA-Werte als gleich erkannt werden
ranking["irradiance_rounded"] = ranking["annual_irradiance"].round(2)

# Doppelte NASA-Werte entfernen
# Pro identischem Einstrahlungswert bleibt nur ein Punkt übrig
ranking_unique = ranking.drop_duplicates(
    subset=["irradiance_rounded"],
    keep="first"
)

ranking_unique = ranking_unique.sort_values(
    "annual_irradiance",
    ascending=False
)

ranking_unique["rank"] = range(1, len(ranking_unique) + 1)

print("Anzahl ursprünglicher Rasterpunkte:", len(ranking))
print("Anzahl eindeutiger NASA-Werte:", len(ranking_unique))

print(ranking_unique.head(20))


# --------------------------------------------------
# 5. Heatmap mit eindeutigen Werten
# --------------------------------------------------

m = folium.Map(
    location=[48.8, 11.5],
    zoom_start=7
)

heat_data = []

for _, row in ranking_unique.iterrows():
    heat_data.append([
        row["lat"],
        row["lon"],
        row["annual_irradiance"]
])

HeatMap(
    heat_data,
    radius=35,
    blur=25,
    max_zoom=10
).add_to(m)


# --------------------------------------------------
# 6. Top-20 Marker hinzufügen
# --------------------------------------------------

top_20 = ranking_unique.head(20)

for _, row in top_20.iterrows():
    folium.Marker(
        location=[row["lat"], row["lon"]],
        popup=(
            f"Rang: {int(row['rank'])}<br>"
            f"Sonneneinstrahlung: {row['annual_irradiance']:.2f} kWh/m²/Jahr<br>"
            f"Lat: {row['lat']}<br>"
            f"Lon: {row['lon']}"
)
).add_to(m)


# --------------------------------------------------
# 7. Karte speichern
# --------------------------------------------------

m.save("bayern_solar_heatmap_unique.html")

m

Lade 47.2, 8.9
Lade 47.2, 9.15
Lade 47.2, 9.4
Lade 47.2, 9.65
Lade 47.2, 9.9
Lade 47.2, 10.15
Lade 47.2, 10.4
Lade 47.2, 10.65
Lade 47.2, 10.9
Lade 47.2, 11.15
Lade 47.2, 11.4
Lade 47.2, 11.65
Lade 47.2, 11.9
Lade 47.2, 12.15
Lade 47.2, 12.4
Lade 47.2, 12.65
Lade 47.2, 12.9
Lade 47.2, 13.15
Lade 47.2, 13.4
Lade 47.2, 13.65
Lade 47.45, 8.9
Lade 47.45, 9.15
Lade 47.45, 9.4
Lade 47.45, 9.65
Lade 47.45, 9.9
Lade 47.45, 10.15
Lade 47.45, 10.4
Lade 47.45, 10.65
Lade 47.45, 10.9
Lade 47.45, 11.15
Lade 47.45, 11.4
Lade 47.45, 11.65
Lade 47.45, 11.9
Lade 47.45, 12.15
Lade 47.45, 12.4
Lade 47.45, 12.65
Lade 47.45, 12.9
Lade 47.45, 13.15
Lade 47.45, 13.4
Lade 47.45, 13.65
Lade 47.7, 8.9
Lade 47.7, 9.15
Lade 47.7, 9.4
Lade 47.7, 9.65
Lade 47.7, 9.9
Lade 47.7, 10.15
Lade 47.7, 10.4
Lade 47.7, 10.65
Lade 47.7, 10.9
Lade 47.7, 11.15
Lade 47.7, 11.4
Lade 47.7, 11.65
Lade 47.7, 11.9
Lade 47.7, 12.15
Lade 47.7, 12.4
Lade 47.7, 12.65
Lade 47.7, 12.9
Lade 47.7, 13.15
Lade 47.7, 13.4
Lade 47.7, 13.65
Lade 